In [3]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

import mlflow 
import mlflow.sklearn

In [1]:
RANDOM_STATE = 42
EXPERIMENT_NAME = "titanic_baseline_training"


In [4]:
mlflow.set_experiment(EXPERIMENT_NAME)

2025/12/30 23:18:25 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/30 23:18:25 INFO mlflow.store.db.utils: Updating database tables
2025/12/30 23:18:25 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/30 23:18:25 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/30 23:18:25 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/30 23:18:25 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location=('file:c:/Users/Aman/OneDrive/Documents/GitHub/ML-Models/ML-EXPERIMENT '
 '1/notebooks/mlruns/2'), creation_time=1767116660959, experiment_id='2', last_update_time=1767116660959, lifecycle_stage='active', name='titanic_baseline_training', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [5]:
X_train = pd.read_csv("../data/X_train.csv")
X_test  = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/y_test.csv").squeeze()

print(X_train.shape, X_test.shape)


(712, 7) (179, 7)


In [7]:
NUMERIC_FEATURES = [
    "Age",
    "Fare",
    "SibSp",
    "Parch"
]

CATEGORICAL_FEATURES = [
    "Sex",
    "Embarked",
    "Pclass",
]

In [8]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES)
    ]
)

In [14]:
rf_model = RandomForestClassifier(
    n_estimators= 100,
    max_depth= None,
    min_samples_split= 2,
    random_state= RANDOM_STATE, 
    n_jobs=1
)

In [15]:
rf_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", rf_model)
])

In [16]:
with mlflow.start_run():

    # ---- Params ----
    mlflow.log_param("model_type", "random_forest")
    mlflow.log_param("n_estimators", rf_model.n_estimators)
    mlflow.log_param("max_depth", rf_model.max_depth)
    mlflow.log_param("min_samples_split", rf_model.min_samples_split)

    # ---- Train ----
    rf_pipeline.fit(X_train, y_train)

    # ---- Inference ----
    y_pred = rf_pipeline.predict(X_test)

    # ---- Metrics ----
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

    # ---- Log model ----
    mlflow.sklearn.log_model(
        sk_model=rf_pipeline,
        artifact_path="model"
    )

2025/12/30 23:22:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [17]:
print("Random Forest Results")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

Random Forest Results
Accuracy  : 0.8101
Precision : 0.7869
Recall    : 0.6957
